# 08 · Business recommendations
Sizing the actions from the analysis.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
k = o('q01_headline_kpis'); k

,period,attempts,successful_transactions,success_rate_pct,value_kes_bn,fee_income_kes_m,fee_per_successful_txn,active_customers,active_rate_pct
0,May 2025 - Oct 2026,"2,022,688.00","1,951,275.00",96.47,29.33,51.72,26.50,NaN,NaN
1,Oct 2025,"98,863.00","95,352.00",96.45,1.43,2.56,26.81,"15,729.00",73.20
2,Oct 2026,"147,378.00","142,328.00",96.57,2.17,3.75,26.36,"21,445.00",71.50


In [3]:
fl = o('q19_fees_lost_to_failures'); tot = fl[fl.channel.isna()].iloc[0]
agent = fl[(fl.channel == 'Agent') & (fl.failure_reason == 'Agent float unavailable')].iloc[0]
ussd = fl[(fl.channel == 'USSD') & fl.failure_reason.isna()].iloc[0]
b = o('q20_fee_billing_exceptions'); fz = o('q22_waiver_fee_foregone')
dormant = o('q24_activity_status'); dm = dormant[dormant.activity_status == 'Dormant (90+ days)']
summary = pd.DataFrame([
    ("Unrecovered operational failures (18 months)", f"{int(tot.not_recovered):,} attempts", f"KES {tot.fees_lost_kes/1e3:,.0f}k fees; KES {tot.value_not_completed_kes_m:,.0f}M not moved"),
    ("  of which agent float shortages", f"{int(agent.not_recovered):,}", f"KES {agent.fees_lost_kes/1e3:,.0f}k fees"),
    ("  of which USSD (all operational reasons)", f"{int(ussd.not_recovered):,}", f"KES {ussd.fees_lost_kes/1e3:,.0f}k fees"),
    ("Fee-engine overcharges to refund", f"{b.transactions.sum():,} transactions / {b.customers_affected.sum():,} customer-cases", f"KES {b.overcharge_kes.sum():,.0f}"),
    ("Wallet-transfer waiver cost (Feb - Oct 2026)", f"{fz.waived_app_transfers.sum():,} free transfers", f"KES {fz.fee_foregone_kes.sum()/1e3:,.0f}k"),
    ("Dormant customers (90+ days)", f"{dm.customers.sum():,}", "re-activation target"),
], columns=['item', 'volume', 'value'])
summary

,item,volume,value
0,Unrecovered operational failures (18 months),"16,822 attempts",KES 633k fees; KES 251M not moved
1,of which agent float shortages,"2,503",KES 212k fees
2,of which USSD (all operational reasons),"5,441",KES 76k fees
3,Fee-engine overcharges to refund,"4,051 transactions / 3,651 customer-cases","KES 96,444"
4,Wallet-transfer waiver cost (Feb - Oct 2026),"58,004 free transfers",KES 562k
5,Dormant customers (90+ days),"5,591",re-activation target


## Recommendations

| # | Recommendation | Evidence | Measure |
|---|---|---|---|
| 1 | **Agent float management for payday and weekends**: float top-up alerts and liquidity support for high-volume agents from the 24th of each month and on Fridays | Float failures 6% on payday days and weekends vs 1% otherwise; largest source of lost fees (q12, q19) | Agent float failure rate below 2% on payday days |
| 2 | **USSD peak-hour capacity**: raise session capacity with the network partner for 17:00–21:00 and keep moving small transfers to the app | USSD operational failures double at the evening peak; USSD is the least reliable channel (q9, q11) | USSD operational failure rate at 17:00–21:00 |
| 3 | **Refund the fee-engine overcharges and add a daily tariff reconciliation** | KES 96k charged against the tariff across about 4,000 transactions in two defects (q20) | Zero tariff variances in the daily check |
| 4 | **Release controls for the app**: staged roll-out and automatic rollback when the timeout rate exceeds twice its normal level | Timeouts rose sevenfold for 10 days after a release; not in the incident log (q15) | Release-related timeout spikes caught within hours |
| 5 | **Log every outage, using anomaly monitoring as a safety net** | The anomaly model found high-failure episodes that are missing from the incident log (nb 07) | Share of anomaly episodes with a matching incident record |
| 6 | **Win back dormant customers and follow up on bad experiences**: reactivation offers for the dormant 18%, and proactive contact after two or more operational failures in a month | Customers with 2+ failures are about twice as likely to be inactive the next month at the same activity level (q28) | 90-day reactivation rate; next-month activity after failures |

**What the data cannot tell us:** why individual customers go dormant (no survey), competitor pricing, the network partner's own capacity data, and costs (fee income is revenue, not profit).